In [1]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv("data/processed/shot_logs_cleaned.csv")

# Shot type classifiers

# 3-pointer indicator
df_clean["three_pointer"] = (df_clean["pts_type"] == 3).astype(int)

# paint proxy (2pt shots <= 8 ft)
df_clean["paint"] = (
    (df_clean["pts_type"] == 2) & (df_clean["shot_dist"] <= 8)
).astype(int)

# mid-range (2pt but not paint)
df_clean["mid_range"] = (
    (df_clean["pts_type"] == 2) & (df_clean["paint"] == 0)
).astype(int)

# catch and shoot
df_clean["catch_and_shoot"] = (
    (df_clean["dribbles"] == 0) & (df_clean["touch_time"] < 2)
).astype(int)

# pull-up shot
df_clean["pull_up"] = (
    (df_clean["dribbles"] >= 1) & (df_clean["touch_time"] >= 2)
).astype(int)

# iso shot proxy
df_clean["iso"] = (
    (df_clean["dribbles"] >= 4) | (df_clean["touch_time"] > 5)
).astype(int)

In [ ]:
# Composite features

# shot clock as share of full clock
df_clean["shot_clock_pct"] = df_clean["shot_clock"] / 24

# game clock as share of elapsed game time scale
df_clean["game_clock_pct"] = df_clean["game_clock_sec"] / (df_clean["period"] * 720)

# defender distance relative to shot distance
df_clean["defender_distance_ratio"] = df_clean["close_def_dist"] / (df_clean["shot_dist"] + 1)

# time pressure from low shot clock
df_clean["time_pressure_index"] = 1 / (df_clean["shot_clock"] + 1)

# early clock shot indicator
df_clean["early_clock_indicator"] = (df_clean["shot_clock"] >= 18).astype(int)

# interaction between defense and shot distance
df_clean["def_dist_x_shot_dist"] = df_clean["close_def_dist"] * df_clean["shot_dist"]

# nonlinear shot distance effect
df_clean["shot_dist_squared"] = df_clean["shot_dist"] ** 2

# log based implementation of self-creation load
df_clean["creation_load_log"] = np.log1p(df_clean["dribbles"] * df_clean["touch_time"])

print("Features engineered. New columns:")
print([c for c in df_clean.columns if c not in [
    "player_name", "matchup", "location", "period", "game_clock_sec",
    "shot_clock", "shot_clock_missing", "dribbles", "touch_time",
    "shot_dist", "close_def_dist", "pts_type", "made"
]])

# save cleaned dataset with all engineered features
df_clean.to_csv("data/processed/shot_logs_cleaned_engineered.csv", index=False)
print("Saved to data/processed/shot_logs_cleaned_engineered.csv")

Features engineered. New columns:
['three_pointer', 'paint', 'mid_range', 'catch_and_shoot', 'pull_up', 'iso', 'shot_clock_pct', 'game_clock_pct', 'defender_distance_ratio', 'time_pressure_index', 'early_clock_indicator', 'def_dist_x_shot_dist', 'shot_dist_squared', 'creation_load_log']
Saved to data/processed/shot_logs_cleaned_engineered.csv
